##DOWNLOAD THE VTUAD INTO THE INPUTS FOLDER
##CREATE A CUSTOM ONC DATASET AND ALSO PLACE INTO INPUTS FOLDER

##ONC DATASET SETTINGS

In [ ]:
#......

##2000_4000 processing

In [ ]:
# This script constructs a split within each class of each ship ID to prevent leakage


import csv
import shutil
from collections import defaultdict
from pathlib import Path


ROOT = Path("/Users/mandeepwalia/Downloads/Vessel-Classification-Representations-Architectures-and-Hyperparameters-main/Inputs/2000_4000") #MAKE SURE TO HAVE THE DATASET INSTALLED IN THIS FOLDER
OUT = Path("Outputs/2000_4000/2000_4000_splits") #MAKE SURE TO PASTE YOUR OUTPUTS PATH BEFORE 2000_4000_splits
SPLITS = ["train", "validation", "test"]
MOVE = False
SPLIT_ORDER = {"train": 0, "validation": 1, "test": 2}


def mmsi_to_folder(mmsi_raw): #Normalize MMSI to integer for clean folder name
    s = str(mmsi_raw).strip()
    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
    except ValueError:
        pass
    return s.replace("/", "_")


def main():
    if not ROOT.is_dir():
        raise SystemExit(f"root not found: {ROOT}")

    # Key: Obtain (MMSI, Class)
    # Value: list of tuples: (split order, file index, and source_wav_path)
    groups = defaultdict(list)
    total_rows = 0
    missing = []

    for split in SPLITS:
        csv_path = ROOT / split / f"metadata_{split}.csv"
        audio_dir = ROOT / split / "audio"
        if not csv_path.is_file():
            raise SystemExit(f"metadata not found: {csv_path}")
        with open(csv_path, newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                total_rows += 1
                cls = row["label"].strip()
                mmsi = mmsi_to_folder(row["MMSI"])
                fidx = row["file_index"].strip()
                src = audio_dir / cls / f"{fidx}.wav"
                if not src.is_file():
                    missing.append(str(src))
                    continue
                try:
                    order_key = (SPLIT_ORDER[split], int(fidx))
                except ValueError:
                    order_key = (SPLIT_ORDER[split], fidx)
                groups[(cls, mmsi)].append((order_key, src))

    print(f"read {total_rows} rows across {len(SPLITS)} splits")
    print(f"found {len(groups)} (class, ship) groups")
    if missing:
        print(f"WARNING: {len(missing)} rows had no matching .wav on disk "
              f"(first few): {missing[:5]}")

    OUT.mkdir(parents=True, exist_ok=True)


    per_class_ships = defaultdict(int)
    per_class_clips = defaultdict(int)
    grand_copied = 0

    for (cls, mmsi), items in sorted(groups.items()):
        items.sort(key=lambda x: x[0])
        dest_dir = OUT / f"{cls}_ship_ids" / f"ship_id_{mmsi}"
        dest_dir.mkdir(parents=True, exist_ok=True)
        for new_idx, (_, src) in enumerate(items):
            dest = dest_dir / f"{mmsi}_{new_idx}.wav"
            if MOVE:
                shutil.move(str(src), str(dest))
            else:
                shutil.copy2(str(src), str(dest))
            grand_copied += 1
        per_class_ships[cls] += 1
        per_class_clips[cls] += len(items)

    print("\n=== summary (per class) ===")
    print(f"{'class':16}{'ships':>7}{'clips':>9}")
    for cls in sorted(per_class_clips):
        print(f"{cls:16}{per_class_ships[cls]:>7}{per_class_clips[cls]:>9}")
    print("-" * 32)
    print(f"{'TOTAL':16}{sum(per_class_ships.values()):>7}{grand_copied:>9}")
    print(f"\n{'moved' if MOVE else 'copied'} {grand_copied} wav files -> {OUT}")


if __name__ == "__main__":
    main()

##Create Train 2000_4000

In [ ]:
#This script partitions ship ids within each class into a training dataset, ensuring no leakage into the validation or test sets.

import random
import shutil
from pathlib import Path


OUT = Path(".....Inputs/2000_4000_2000_4000_splits") #CHANGE THIS
TRAIN_DIR = Path("...OUTPUTS/2000_4000/train")#CHANGE THIS
SEED = 42

TARGETS = {
    "background": 688,
    "cargo": 688,
    "passengership": 688,
    "tanker": 688,
    "tug": 688,
}
BACKGROUND_CLASS = "background"


def list_ship_dirs(class_dir):
    if not class_dir.is_dir():
        return []
    ships = [d for d in sorted(class_dir.iterdir())
             if d.is_dir() and d.name.startswith("ship_id_") and not d.name.endswith("_used")]
    return ships


def wavs_in(ship_dir):
    return sorted(p for p in ship_dir.iterdir()
                  if p.suffix.lower() == ".wav" and not p.stem.endswith("_used"))


def mark_file_used(p):
    target = p.with_name(p.stem + "_used.wav")
    if not target.exists():
        p.rename(target)


def build_background(class_dir, dest_dir, target, rng):
    ships = list_ship_dirs(class_dir)
    if not ships:
        print(f"[background] no available ship folders in {class_dir} -> skip", flush=True)
        return 0

    all_wavs = []
    for s in ships:
        all_wavs.extend(wavs_in(s))
    if not all_wavs:
        print(f"[background] no unused wavs in {class_dir} -> skip", flush=True)
        return 0

    pool_size = len(all_wavs) // 3          # 1/3 for train, 1/3 val, 1/3 test
    if pool_size < 1:
        pool_size = 1
    pool = rng.sample(all_wavs, pool_size)
    rng.shuffle(pool)

    print(f"[background] {len(ships)} ship id(s), {len(all_wavs)} unused clips -> "
          f"file pool floor({len(all_wavs)}/3)={pool_size}, target {target}", flush=True)

    take = min(target, len(pool))
    for p in pool[:take]:
        shutil.copy2(str(p), str(dest_dir / p.name))


    for p in pool:
        mark_file_used(p)

    status = "OK" if take == target else f"SHORT by {target - take}"
    print(f"    copied {take}/{target} clips from the file pool "
          f"({pool_size} files marked _used) [{status}]", flush=True)
    if take < target:
        print(f"    !! the 1/3 file pool holds only {pool_size} clips (< {target})", flush=True)
    return take


def main():
    rng = random.Random(SEED)
    if not OUT.is_dir():
        print(f"source outputs dir not found: {OUT}. Creating automatically")
        OUT.mkdir(parents=True, exist_ok=True)

    TRAIN_DIR.mkdir(parents=True, exist_ok=True)
    print(f"SEED = {SEED}\n", flush=True)

    grand_total = 0
    for cls, target in TARGETS.items():
        class_dir = OUT / f"{cls}_ship_ids"
        dest_dir = TRAIN_DIR / cls
        dest_dir.mkdir(parents=True, exist_ok=True)


        if cls == BACKGROUND_CLASS:
            grand_total += build_background(class_dir, dest_dir, target, rng)
            continue

        ships = list_ship_dirs(class_dir)
        if not ships:
            print(f"[{cls}] no available ship folders in {class_dir} -> skip", flush=True)
            continue


        n_ships = len(ships) // 3 #partition by 3, 1/3 for train, 1/3 for validation, 1/3 for test
        if n_ships < 1:
            n_ships = 1
        n_ships = min(n_ships, len(ships))

        chosen = rng.sample(ships, n_ships)            # random ship selection using random seed
        quota = target // n_ships

        # pre-load and shuffle each chosen ship's clips
        pool = {}
        for s in chosen:
            w = wavs_in(s)
            rng.shuffle(w)
            pool[s] = w

        print(f"[{cls}] {len(ships)} ships available -> using {n_ships} "
              f"(floor quota {quota}/ship), target {target}", flush=True)

        copied = 0
        taken = {s: 0 for s in chosen}

        # pass 1: take the floor quota from each chosen ship
        for s in chosen:
            avail = pool[s]
            take = min(quota, len(avail), target - copied)
            for p in avail[taken[s]: taken[s] + take]:
                shutil.copy2(str(p), str(dest_dir / p.name))
            taken[s] += take
            copied += take
            if copied >= target:
                break

        #We take whatever is left in other ship ids until target is met
        if copied < target:
            for s in chosen:
                if copied >= target:
                    break
                avail = pool[s]
                remaining_in_ship = avail[taken[s]:]
                need = target - copied
                take = min(need, len(remaining_in_ship))
                for p in remaining_in_ship[:take]:
                    shutil.copy2(str(p), str(dest_dir / p.name))
                taken[s] += take
                copied += take

        # mark each consumed ship folder as used
        for s in chosen:
            used_name = s.parent / f"{s.name}_used"
            if not used_name.exists():
                s.rename(used_name)

        status = "OK" if copied == target else f"SHORT by {target - copied}"
        print(f"    copied {copied}/{target} clips from {n_ships} ships "
              f"[{status}]", flush=True)
        if copied < target:
            print(f"    !! not enough clips across the {n_ships} selected ships to "
                  f"reach {target}; consider allocating more ships to this class", flush=True)
        grand_total += copied

    print(f"\nTRAIN dataset written to {TRAIN_DIR}")
    print(f"total clips copied: {grand_total}")


if __name__ == "__main__":
    main()

Val for 2000_4000

In [ ]:
#This script partitions ship ids within each class into a validation dataset, drawing only
#from ships/files not already consumed by the train build, so there is no
#leakage between train, validation, and test.

import random
import shutil
from pathlib import Path


OUT = Path(".....Inputs/2000_4000_2000_4000_splits") #CHANGE THIS
VAL_DIR = Path("...OUTPUTS/2000_4000/validation")#CHANGE THIS
SEED = 42
DIVISOR = 2

TARGETS = {                        # required clip count per class in the validation set
    "background": 72,
    "cargo": 72,
    "passengership": 72,
    "tanker": 72,
    "tug": 72,
}
BACKGROUND_CLASS = "background"    # one ship id


def list_ship_dirs(class_dir):
    if not class_dir.is_dir():
        return []
    ships = [d for d in sorted(class_dir.iterdir())
             if d.is_dir() and d.name.startswith("ship_id_") and not d.name.endswith("_used")]
    return ships


def wavs_in(ship_dir):
    return sorted(p for p in ship_dir.iterdir()
                  if p.suffix.lower() == ".wav" and not p.stem.endswith("_used"))


def mark_file_used(p):
    target = p.with_name(p.stem + "_used.wav")
    if not target.exists():
        p.rename(target)


def build_background(class_dir, dest_dir, target, rng):
    ships = list_ship_dirs(class_dir)
    if not ships:
        print(f"[background] no available ship folders in {class_dir} -> skip", flush=True)
        return 0

    all_wavs = []
    for s in ships:
        all_wavs.extend(wavs_in(s))          # excludes train's _used files
    if not all_wavs:
        print(f"[background] no unused wavs left in {class_dir} -> skip", flush=True)
        return 0

    pool_size = len(all_wavs) // DIVISOR
    if pool_size < 1:
        pool_size = 1
    pool = rng.sample(all_wavs, pool_size)
    rng.shuffle(pool)

    print(f"[background] {len(ships)} ship id(s), {len(all_wavs)} unused clips -> "
          f"file pool floor({len(all_wavs)}/{DIVISOR})={pool_size}, target {target}", flush=True)

    take = min(target, len(pool))
    for p in pool[:take]:
        shutil.copy2(str(p), str(dest_dir / p.name))

    for p in pool:                            # mark the half used
        mark_file_used(p)

    status = "OK" if take == target else f"SHORT by {target - take}"
    print(f"    copied {take}/{target} clips from the file pool "
          f"({pool_size} files marked _used) [{status}]", flush=True)
    if take < target:
        print(f"    !! the 1/{DIVISOR} file pool holds only {pool_size} clips "
              f"(< {target})", flush=True)
    return take


def main():
    rng = random.Random(SEED)
    if not OUT.is_dir():
        raise SystemExit(f"source outputs dir not found: {OUT}")
    VAL_DIR.mkdir(parents=True, exist_ok=True)
    print(f"SEED = {SEED}\n", flush=True)

    grand_total = 0
    for cls, target in TARGETS.items():
        class_dir = OUT / f"{cls}_ship_ids"
        dest_dir = VAL_DIR / cls
        dest_dir.mkdir(parents=True, exist_ok=True)

        # background: single ship id
        if cls == BACKGROUND_CLASS:
            grand_total += build_background(class_dir, dest_dir, target, rng)
            continue

        ships = list_ship_dirs(class_dir)      # excludes train's _used ships
        if not ships:
            print(f"[{cls}] no available ship folders in {class_dir} -> skip", flush=True)
            continue

        # how many ships to use for validation
        n_ships = len(ships) // DIVISOR
        if n_ships < 1:
            n_ships = 1
        n_ships = min(n_ships, len(ships))

        chosen = rng.sample(ships, n_ships)            # random ship selection using random seed
        quota = target // n_ships

        # pre-load and shuffle each chosen ship's clips
        pool = {}
        for s in chosen:
            w = wavs_in(s)
            rng.shuffle(w)
            pool[s] = w

        print(f"[{cls}] {len(ships)} unused ships -> using {n_ships} "
              f"(floor quota {quota}/ship), target {target}", flush=True)

        copied = 0
        taken = {s: 0 for s in chosen}

        # pass 1: take the floor quota from each chosen ship
        for s in chosen:
            avail = pool[s]
            take = min(quota, len(avail), target - copied)
            for p in avail[taken[s]: taken[s] + take]:
                shutil.copy2(str(p), str(dest_dir / p.name))
            taken[s] += take
            copied += take
            if copied >= target:
                break

        # pass 2: fill the remaining shortfall from the other chosen ships,
        #We take whatever is left in other ship ids until target is met
        if copied < target:
            for s in chosen:
                if copied >= target:
                    break
                avail = pool[s]
                remaining_in_ship = avail[taken[s]:]
                need = target - copied
                take = min(need, len(remaining_in_ship))
                for p in remaining_in_ship[:take]:
                    shutil.copy2(str(p), str(dest_dir / p.name))
                taken[s] += take
                copied += take

        # mark each consumed ship folder as used
        for s in chosen:
            used_name = s.parent / f"{s.name}_used"
            if not used_name.exists():
                s.rename(used_name)

        status = "OK" if copied == target else f"SHORT by {target - copied}"
        print(f"    copied {copied}/{target} clips from {n_ships} ships "
              f"[{status}]", flush=True)
        if copied < target:
            print(f"    !! not enough clips across the {n_ships} selected ships to "
                  f"reach {target}; consider allocating more ships to this class", flush=True)
        grand_total += copied

    print(f"\nVALIDATION dataset written to {VAL_DIR}")
    print(f"total clips copied: {grand_total}")


if __name__ == "__main__":
    main()

##TEST for 2000_4000

In [ ]:
#This script partitions the remaining ship ids within each class into a test dataset. It draws
#only from ships/files not already consumed by the train and validation builds,
#completing a leakage-free three-way split.

import random
import shutil
from pathlib import Path


OUT = Path(".....Inputs/2000_4000_2000_4000_splits") #CHANGE THIS
TEST_DIR = Path("...OUTPUTS/2000_4000/test")#CHANGE THIS
SEED = 42
DIVISOR = 1                        # final split

TARGETS = {                        # required clip count per class in the test set
    "background": 40,
    "cargo": 40,
    "passengership": 40,
    "tanker": 40,
    "tug": 40,
}
BACKGROUND_CLASS = "background"    # one ship id


def list_ship_dirs(class_dir):
    """Unused ship_id_* folders in a class folder (skips ones already marked _used)."""
    if not class_dir.is_dir():
        return []
    ships = [d for d in sorted(class_dir.iterdir())
             if d.is_dir() and d.name.startswith("ship_id_") and not d.name.endswith("_used")]
    return ships


def wavs_in(ship_dir):
    """Unused wavs in a ship folder (skips files already marked _used)."""
    return sorted(p for p in ship_dir.iterdir()
                  if p.suffix.lower() == ".wav" and not p.stem.endswith("_used"))


def mark_file_used(p):
    """Rename a wav to <stem>_used.wav (records what this split consumed)."""
    target = p.with_name(p.stem + "_used.wav")
    if not target.exists():
        p.rename(target)


def build_background(class_dir, dest_dir, target, rng):
    """Background has a single ship id, so its FILES are partitioned instead of its ships.
    Train and validation already marked their thirds/halves _used; test uses ALL remaining
    files, draws the target from them, and marks them _used."""
    ships = list_ship_dirs(class_dir)
    if not ships:
        print(f"[background] no available ship folders in {class_dir} -> skip", flush=True)
        return 0

    all_wavs = []
    for s in ships:
        all_wavs.extend(wavs_in(s))
    if not all_wavs:
        print(f"[background] no unused wavs left in {class_dir} -> skip", flush=True)
        return 0

    pool_size = len(all_wavs) // DIVISOR
    if pool_size < 1:
        pool_size = 1
    pool = rng.sample(all_wavs, pool_size)
    rng.shuffle(pool)

    print(f"[background] {len(ships)} ship id(s), {len(all_wavs)} unused clips -> "
          f"file pool (all) {pool_size}, target {target}", flush=True)

    take = min(target, len(pool))
    for p in pool[:take]:
        shutil.copy2(str(p), str(dest_dir / p.name))

    for p in pool:
        mark_file_used(p)

    status = "OK" if take == target else f"SHORT by {target - take}"
    print(f"    copied {take}/{target} clips from the file pool "
          f"({pool_size} files marked _used) [{status}]", flush=True)
    if take < target:
        print(f"    !! only {pool_size} clips remained (< {target}); this was the last split",
              flush=True)
    return take


def main():
    rng = random.Random(SEED)
    if not OUT.is_dir():
        raise SystemExit(f"source outputs dir not found: {OUT}")
    TEST_DIR.mkdir(parents=True, exist_ok=True)
    print(f"SEED = {SEED}\n", flush=True)

    grand_total = 0
    for cls, target in TARGETS.items():
        class_dir = OUT / f"{cls}_ship_ids"
        dest_dir = TEST_DIR / cls
        dest_dir.mkdir(parents=True, exist_ok=True)

        # background: single ship id
        if cls == BACKGROUND_CLASS:
            grand_total += build_background(class_dir, dest_dir, target, rng)
            continue

        ships = list_ship_dirs(class_dir)
        if not ships:
            print(f"[{cls}] no available ship folders in {class_dir} -> skip", flush=True)
            continue

        # how many ships to use for test
        n_ships = len(ships) // DIVISOR
        if n_ships < 1:
            n_ships = 1
        n_ships = min(n_ships, len(ships))

        chosen = rng.sample(ships, n_ships)            # random ship selection using random seed
        quota = target // n_ships

        # pre-load and shuffle each chosen ship's clips
        pool = {}
        for s in chosen:
            w = wavs_in(s)
            rng.shuffle(w)
            pool[s] = w

        print(f"[{cls}] {len(ships)} unused ships -> using {n_ships} "
              f"(floor quota {quota}/ship), target {target}", flush=True)

        copied = 0
        taken = {s: 0 for s in chosen}

        # pass 1: take the floor quota from each chosen ship
        for s in chosen:
            avail = pool[s]
            take = min(quota, len(avail), target - copied)
            for p in avail[taken[s]: taken[s] + take]:
                shutil.copy2(str(p), str(dest_dir / p.name))
            taken[s] += take
            copied += take
            if copied >= target:
                break

        # pass 2: fill the remaining shortfall from the other chosen ships,
        #We take whatever is left in other ship ids until target is met
        if copied < target:
            for s in chosen:
                if copied >= target:
                    break
                avail = pool[s]
                remaining_in_ship = avail[taken[s]:]
                need = target - copied
                take = min(need, len(remaining_in_ship))
                for p in remaining_in_ship[:take]:
                    shutil.copy2(str(p), str(dest_dir / p.name))
                taken[s] += take
                copied += take

        # mark each consumed ship folder as used
        for s in chosen:
            used_name = s.parent / f"{s.name}_used"
            if not used_name.exists():
                s.rename(used_name)

        status = "OK" if copied == target else f"SHORT by {target - copied}"
        print(f"    copied {copied}/{target} clips from {n_ships} ships "
              f"[{status}]", flush=True)
        if copied < target:
            print(f"    !! not enough clips across the {n_ships} remaining ships to "
                  f"reach {target}; this was the last split", flush=True)
        grand_total += copied

    print(f"\nTEST dataset written to {TEST_DIR}")
    print(f"total clips copied: {grand_total}")


if __name__ == "__main__":
    main()

##REPEAT THE STEPS FOR 3000_5000 AND 4000_6000, JUST MAKE SURE TO CHANGE THE FILE PATHS!

#ONC SETTINGS: (config.py)



First csv file:

begin,end,latitude,longitude,depth,location
2016-05-02T15:24:11.000Z,2016-07-24T08:33:27.000Z,49.080927,-123.338713,141.0,LSBBL

In [ ]:
MAX_INCLUSION_RADIUS=15000.0
INCLUSION_RADIUS=2000

UNIQUE_SCENARIOS=True

METADATA_SECONDS=1
METADATA_FILE="metadata"
METADATA_VAL_SPLIT=0.09
METADATA_TEST_SPLIT=0.05

# Pacific - Salish Sea - Strait of Georgia - Fraser River Delta (49.080927,-123.338713)
AIS_CODE = "DIGITALYACHTAISNET1302-0097-01"
WAV_DEVICES = [{"deviceCode": "ICLISTENAF2523"}, {"deviceCode": "ICLISTENAF2556"}]
CTD_DEVICE = "SBECTD19p6935"

# Define if the metadata will include ctd information. Only needed for step 10.
USE_CTD=False

##SPLIT SHIP IDS FOR ONC DATASET

In [ ]:
#This script groups the classified clips into class/MMSI folders and writes a
#manifest + summary CSV describing what was placed where.

import os
import shutil
import wave

import pandas as pd

from tqdm import tqdm


# ======================= EDIT THESE =======================
METADATA_ROOT = "....../07b_classified_wav_files/inclusion_2000_exclusion_4000"  #CHANGE THIS: folder holding the metadata CSV; clip paths in the CSV are relative to it
METADATA_FILE = "metadata_1s.csv"                                                #CHANGE THIS: metadata CSV file name inside METADATA_ROOT
OUTPUT_DIR = "Outputs/ONC/ship_splits"        #CHANGE THIS: where the class/MMSI folders will be written

UNIT = "file"       # "file" = place whole WAV files, "segment" = cut into fixed length segments
MODE = "hardlink"   # "hardlink", "symlink" or "copy" — only used when UNIT is "file"
SECONDS = 1         # duration of each segment — only used when UNIT is "segment"
# ==========================================================


BACKGROUND_LABEL = "background"
# The background class has no vessel identity, so every background clip is
# stored under a single MMSI folder.
BACKGROUND_MMSI = 0


class bcolors:
    HEADER = "\033[95m"
    WARNING = "\033[93m"
    ENDC = "\033[0m"


def create_dir(parent, name):
    directory = os.path.join(parent, name)
    os.makedirs(directory, exist_ok=True)
    return directory


def get_mmsi_folder_name(mmsi):
    return str(int(mmsi))


def get_clips_from_metadata(metadata_file):
    """Categorise every clip in the metadata by its class and MMSI.

    Returns one row per WAV file with the label, the MMSI folder name it
    belongs to and the 1 second offsets that the metadata lists for it.
    """
    metadata = pd.read_csv(metadata_file)

    clips = []
    for path, rows in metadata.groupby("path"):
        labels = rows.label.unique()
        mmsis = rows.MMSI.unique()

        if len(labels) > 1:
            print(f"{bcolors.WARNING}Skipping {path}: more than one label {labels}{bcolors.ENDC}")
            continue
        if len(mmsis) > 1:
            print(f"{bcolors.WARNING}Skipping {path}: more than one MMSI {mmsis}{bcolors.ENDC}")
            continue

        label = labels[0]
        mmsi = BACKGROUND_MMSI if label == BACKGROUND_LABEL else mmsis[0]

        clips.append(
            {
                "label": label,
                "mmsi": get_mmsi_folder_name(mmsi),
                "path": path,
                "sub_inits": sorted(rows.sub_init.tolist()),
            }
        )

    return clips


def place_wav_file(source_file, destination_file, mode):
    if os.path.exists(destination_file):
        os.remove(destination_file)

    if mode == "hardlink":
        os.link(source_file, destination_file)
    elif mode == "symlink":
        os.symlink(os.path.abspath(source_file), destination_file)
    else:
        shutil.copyfile(source_file, destination_file)


def save_wav_segments(source_file, destination_directory, sub_inits, seconds):
    """Cut the clip into the fixed length segments listed in the metadata."""
    saved = []
    file_name = os.path.splitext(os.path.basename(source_file))[0]

    with wave.open(source_file, "rb") as source_wav:
        frame_rate = source_wav.getframerate()
        segment_frames = frame_rate * seconds
        total_frames = source_wav.getnframes()

        for sub_init in sub_inits:
            start_frame = sub_init * frame_rate
            if start_frame + segment_frames > total_frames:
                print(
                    f"{bcolors.WARNING}Skipping {source_file} at {sub_init}s: "
                    f"beyond the end of the file{bcolors.ENDC}"
                )
                continue

            source_wav.setpos(start_frame)
            frames = source_wav.readframes(segment_frames)

            destination_file = os.path.join(
                destination_directory, f"{file_name}_{sub_init:05d}.wav"
            )
            with wave.open(destination_file, "wb") as destination_wav:
                destination_wav.setnchannels(source_wav.getnchannels())
                destination_wav.setsampwidth(source_wav.getsampwidth())
                destination_wav.setframerate(frame_rate)
                destination_wav.writeframes(frames)

            saved.append((destination_file, sub_init))

    return saved


def group_clips_by_mmsi(metadata_root, metadata_file, output_directory, unit, mode, seconds):
    clips = get_clips_from_metadata(os.path.join(metadata_root, metadata_file))
    print(f"Categorised {len(clips)} clips by class and MMSI")

    manifest = []
    for clip in tqdm(clips, total=len(clips)):
        # The metadata stores the clip paths relative to the metadata folder.
        source_file = os.path.join(metadata_root, clip["path"])
        if not os.path.exists(source_file):
            print(f"{bcolors.WARNING}Skipping {clip['path']}: file not found{bcolors.ENDC}")
            continue

        # Every MMSI folder lives inside the folder of its own class.
        label_directory = create_dir(output_directory, clip["label"])
        mmsi_directory = create_dir(label_directory, clip["mmsi"])

        if unit == "segment":
            saved = save_wav_segments(source_file, mmsi_directory, clip["sub_inits"], seconds)
        else:
            destination_file = os.path.join(mmsi_directory, os.path.basename(source_file))
            place_wav_file(source_file, destination_file, mode)
            saved = [(destination_file, None)]

        for destination_file, sub_init in saved:
            manifest.append(
                {
                    "label": clip["label"],
                    "mmsi": clip["mmsi"],
                    "group_id": f"{clip['label']}/{clip['mmsi']}",
                    "source_path": clip["path"],
                    "sub_init": sub_init,
                    "path": os.path.relpath(destination_file, output_directory),
                }
            )

    return pd.DataFrame(manifest)


def save_manifest(manifest, output_directory):
    manifest_file = os.path.join(output_directory, "grouped_manifest.csv")
    manifest.to_csv(manifest_file, index=False)

    summary = (
        manifest.groupby("label")
        .agg(mmsi_folders=("mmsi", "nunique"), clips=("path", "count"))
        .sort_values("clips", ascending=False)
    )
    summary_file = os.path.join(output_directory, "grouped_summary.csv")
    summary.to_csv(summary_file)

    print(f"\n{summary.to_string()}")
    print(f"\nTotal MMSI folders: {manifest.group_id.nunique()}")
    print(f"Manifest saved to {manifest_file}")
    print(f"Summary saved to {summary_file}")


def main():
    if not os.path.isdir(METADATA_ROOT):
        raise SystemExit(f"metadata root not found: {METADATA_ROOT}")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print(f"\n{bcolors.HEADER}Grouping clips by class and MMSI{bcolors.ENDC}")
    print(f"Reading {os.path.join(METADATA_ROOT, METADATA_FILE)}")
    print(f"Writing to {OUTPUT_DIR}")

    manifest = group_clips_by_mmsi(
        METADATA_ROOT,
        METADATA_FILE,
        OUTPUT_DIR,
        UNIT,
        MODE,
        SECONDS,
    )
    save_manifest(manifest, OUTPUT_DIR)


if __name__ == "__main__":
    main()


##CREATE TRAIN-VAL-TEST SPLIT

In [ ]:
#This script builds one train/validation/test split from the class/MMSI grouped
#clips. The units it draws from are marked _used so the splits built afterwards
#never reuse the same ship. Run it once per split, changing SPLIT each time
#(train first, then validation, then test).

import os
import wave

import pandas as pd


# ======================= EDIT THESE =======================
GROUPED_DIR = "Outputs/ONC/ship_splits"                                          #CHANGE THIS: the OUTPUT_DIR of the previous cell (holds the class/MMSI folders and grouped_manifest.csv)
METADATA_CSV = "....../07b_classified_wav_files/inclusion_2000_exclusion_4000/metadata_1s.csv"  #CHANGE THIS: full path to the metadata CSV
SPLITS_DIR = "Outputs/ONC/dataset_splits"                                        #CHANGE THIS: where the split folders and manifests will be written

SPLIT = "train"     # which split to build: "train", "validation" or "test". TYPE IN ALL THREE SPLITS
PARTITIONS = None   # how many partitions the available units are divided into; None = the default for the chosen split (train 3, validation 2, test 1)
SECONDS = 1         # duration of each segment
# ==========================================================


BACKGROUND_LABEL = "background"
USED_MARKER = "_used"

# The number of segments that each class contributes to every split.
SPLIT_QUOTAS = {
    "train": {
        "background": 1376,
        "cargo": 688,
        "passengership": 688,
        "tanker": 688,
        "tug": 688,
    },
    "validation": {
        "background": 144,
        "cargo": 72,
        "passengership": 72,
        "tanker": 72,
        "tug": 72,
    },
    "test": {
        "background": 80,
        "cargo": 40,
        "passengership": 40,
        "tanker": 40,
        "tug": 40,
    },
}

# How many partitions the available units are divided into for every split. The
# split is drawn from the first partition and the rest is kept for the splits
# that are built afterwards.
SPLIT_PARTITIONS = {
    "train": 3,
    "validation": 2,
    # The test split is the last one, so there is nothing left to reserve.
    "test": 1,
}


class bcolors:
    HEADER = "\033[95m"
    OKBLUE = "\033[94m"
    WARNING = "\033[93m"
    FAIL = "\033[91m"
    ENDC = "\033[0m"


def create_dir(parent, name):
    directory = os.path.join(parent, name)
    os.makedirs(directory, exist_ok=True)
    return directory


def is_used(name):
    # The marker sits on the folder name for vessels and before the extension
    # for the background clips.
    return os.path.splitext(name)[0].endswith(USED_MARKER)


def get_sort_key(unit):
    """Sort the units numerically so the partitions are reproducible."""
    stem = os.path.splitext(os.path.basename(unit))[0]
    return (0, int(stem)) if stem.isdigit() else (1, unit)


def get_units_from_manifest(manifest, metadata):
    """Build the pool of drawable units for every class.

    The vessel classes are drawn from their MMSI folders. The background class
    has a single MMSI folder, so it is partitioned by clip instead.
    """
    segments = (
        metadata.sort_values(["path", "sub_init"])
        .groupby("path")
        .sub_init.apply(list)
        .to_dict()
    )

    units = {}
    for label, rows in manifest.groupby("label"):
        pool = {}
        for row in rows.itertuples(index=False):
            if any(is_used(part) for part in row.path.split(os.sep)):
                continue

            unit = os.path.basename(row.path) if label == BACKGROUND_LABEL else row.mmsi
            for sub_init in segments.get(row.source_path, []):
                pool.setdefault(str(unit), []).append((row.path, sub_init))

        units[label] = dict(sorted(pool.items(), key=lambda item: get_sort_key(item[0])))

    return units


def draw_evenly(pool, ordered_units, quota, cursors, drawn):
    """Take one segment at a time from each unit in turn until the quota is met."""
    while len(drawn) < quota:
        progressed = False
        for unit in ordered_units:
            if len(drawn) >= quota:
                break
            cursor = cursors[unit]
            if cursor < len(pool[unit]):
                path, sub_init = pool[unit][cursor]
                drawn.append((unit, path, sub_init))
                cursors[unit] = cursor + 1
                progressed = True
        if not progressed:
            break

    return drawn


def draw_from_class(pool, quota, partition_count):
    """Draw the quota from the first partition of units, borrowing if needed."""
    names = list(pool.keys())
    size = len(names) // partition_count
    partitions = [names[index * size : (index + 1) * size] for index in range(partition_count - 1)]
    partitions.append(names[(partition_count - 1) * size :])

    cursors = {unit: 0 for unit in names}
    drawn = []
    ordered = []
    for index, partition in enumerate(partitions):
        if not partition:
            continue
        ordered = ordered + partition
        draw_evenly(pool, ordered, quota, cursors, drawn)
        if len(drawn) >= quota:
            break
        if index < len(partitions) - 1:
            print(
                f"{bcolors.WARNING}  quota not met from partition {index + 1}, "
                f"borrowing from partition {index + 2}{bcolors.ENDC}"
            )

    return drawn, size


def save_segments(grouped_directory, output_directory, label, drawn, seconds):
    """Cut and save every drawn segment, opening each source clip only once."""
    by_clip = {}
    for unit, path, sub_init in drawn:
        by_clip.setdefault((unit, path), []).append(sub_init)

    saved = []
    for (unit, path), sub_inits in by_clip.items():
        source_file = os.path.join(grouped_directory, path)
        destination_directory = create_dir(
            create_dir(output_directory, label), unit.replace(".wav", "")
        )
        file_name = os.path.splitext(os.path.basename(path))[0]

        with wave.open(source_file, "rb") as source_wav:
            frame_rate = source_wav.getframerate()
            segment_frames = frame_rate * seconds

            for sub_init in sorted(sub_inits):
                source_wav.setpos(sub_init * frame_rate)
                frames = source_wav.readframes(segment_frames)

                destination_file = os.path.join(
                    destination_directory, f"{file_name}_{sub_init:05d}.wav"
                )
                with wave.open(destination_file, "wb") as destination_wav:
                    destination_wav.setnchannels(source_wav.getnchannels())
                    destination_wav.setsampwidth(source_wav.getsampwidth())
                    destination_wav.setframerate(frame_rate)
                    destination_wav.writeframes(frames)

                saved.append(
                    {
                        "label": label,
                        "unit": unit,
                        "source_path": path,
                        "sub_init": sub_init,
                        "path": os.path.relpath(destination_file, output_directory),
                    }
                )

    return saved


def mark_used(grouped_directory, label, used_units):
    """Rename every unit that was drawn from so it is never reused."""
    renamed = {}
    for unit in used_units:
        if label == BACKGROUND_LABEL:
            matches = [
                os.path.join(root, unit)
                for root, _, files in os.walk(os.path.join(grouped_directory, label))
                if unit in files
            ]
            if not matches:
                continue
            current = matches[0]
            stem, extension = os.path.splitext(current)
            target = f"{stem}{USED_MARKER}{extension}"
        else:
            current = os.path.join(grouped_directory, label, unit)
            target = f"{current}{USED_MARKER}"

        if os.path.exists(current) and not os.path.exists(target):
            os.rename(current, target)
        renamed[os.path.relpath(current, grouped_directory)] = os.path.relpath(
            target, grouped_directory
        )

    return renamed


def build_split(
    grouped_directory, metadata_file, output_directory, quotas, partition_count, seconds
):
    manifest = pd.read_csv(os.path.join(grouped_directory, "grouped_manifest.csv"))
    manifest["mmsi"] = manifest.mmsi.astype(str)
    metadata = pd.read_csv(metadata_file)

    units = get_units_from_manifest(manifest, metadata)

    split = []
    renamed = {}
    report = []
    for label, quota in quotas.items():
        pool = units.get(label, {})
        if not pool:
            print(f"{bcolors.FAIL}No units available for {label}{bcolors.ENDC}")
            continue

        print(f"\n{bcolors.OKBLUE}{label}{bcolors.ENDC}: {len(pool)} units, quota {quota}")
        drawn, partition_size = draw_from_class(pool, quota, partition_count)
        if len(drawn) < quota:
            print(
                f"{bcolors.FAIL}  only {len(drawn)} of {quota} segments available"
                f"{bcolors.ENDC}"
            )

        used_units = sorted({unit for unit, _, _ in drawn}, key=get_sort_key)
        saved = save_segments(grouped_directory, output_directory, label, drawn, seconds)
        split.extend(saved)
        renamed.update(mark_used(grouped_directory, label, used_units))

        borrowed = len(used_units) - min(len(used_units), partition_size)
        print(
            f"  drew {len(drawn)} segments from {len(used_units)} units "
            f"(first partition = {partition_size}, borrowed = {borrowed})"
        )
        report.append(
            {
                "label": label,
                "quota": quota,
                "segments": len(drawn),
                "units_total": len(pool),
                "units_in_first_partition": partition_size,
                "units_used": len(used_units),
                "units_borrowed": borrowed,
                "segments_per_unit_min": min(
                    (sum(1 for u, _, _ in drawn if u == unit) for unit in used_units),
                    default=0,
                ),
                "segments_per_unit_max": max(
                    (sum(1 for u, _, _ in drawn if u == unit) for unit in used_units),
                    default=0,
                ),
            }
        )

    # Keep the grouped manifest pointing at the renamed folders.
    if renamed:
        manifest["path"] = [
            next((v for k, v in renamed.items() if p == k or p.startswith(k + os.sep)), p)
            for p in manifest.path
        ]
        manifest.to_csv(os.path.join(grouped_directory, "grouped_manifest.csv"), index=False)

    return pd.DataFrame(split), pd.DataFrame(report)


def main():
    if SPLIT not in SPLIT_QUOTAS:
        raise SystemExit(f"SPLIT must be one of {list(SPLIT_QUOTAS)}, got {SPLIT!r}")
    if not os.path.isdir(GROUPED_DIR):
        raise SystemExit(f"grouped clips dir not found: {GROUPED_DIR}")
    if not os.path.isfile(METADATA_CSV):
        raise SystemExit(f"metadata csv not found: {METADATA_CSV}")

    os.makedirs(SPLITS_DIR, exist_ok=True)
    output_directory = create_dir(SPLITS_DIR, SPLIT)

    partition_count = PARTITIONS or SPLIT_PARTITIONS[SPLIT]

    print(f"\n{bcolors.HEADER}Building the {SPLIT} split{bcolors.ENDC}")
    print(f"Reading {GROUPED_DIR}")
    print(f"Writing to {output_directory}")
    print(f"Dividing the available units into {partition_count} partitions")

    split, report = build_split(
        GROUPED_DIR,
        METADATA_CSV,
        output_directory,
        SPLIT_QUOTAS[SPLIT],
        partition_count,
        SECONDS,
    )

    manifest_file = os.path.join(SPLITS_DIR, f"{SPLIT}_manifest.csv")
    split.to_csv(manifest_file, index=False)
    report.to_csv(os.path.join(SPLITS_DIR, f"{SPLIT}_report.csv"), index=False)

    print(f"\n{report.to_string(index=False)}")
    print(f"\nTotal {SPLIT} segments: {len(split)}")
    print(f"Manifest saved to {manifest_file}")


if __name__ == "__main__":
    main()


##CREATE CUMULATIVE DATASET

In [ ]:
import os
import shutil

SOURCE_DATASETS = ['2000_4000', '3000_5000', '4000_6000', 'ONC']


def create_cumulative_dataset(source_root, output_root=None):
    output_root = source_root + "/cumulative_dataset"
    splits = ['train', 'test', 'validation']

    for split in splits:
        os.makedirs(os.path.join(output_root, split), exist_ok=True)

    dataset_folders = [d for d in SOURCE_DATASETS
                       if os.path.isdir(os.path.join(source_root, d))]
    missing = [d for d in SOURCE_DATASETS if d not in dataset_folders]
    if missing:
        print(f"WARNING: source folders not found: {missing}")
    if not dataset_folders:
        raise SystemExit(f"none of {SOURCE_DATASETS} found under {source_root}")

    print(f"Processing datasets: {dataset_folders}")

    total_copied = 0
    per_dataset = {}
    per_class = {}

    for dataset_idx, dataset_name in enumerate(dataset_folders):
        print(f"  Processing dataset {dataset_idx + 1}/{len(dataset_folders)}: "
              f"{dataset_name}")
        dataset_path = os.path.join(source_root, dataset_name)
        dataset_total = 0

        for split in splits:
            split_path = os.path.join(dataset_path, split)
            if not os.path.exists(split_path):
                print(f"    Skipping split '{split}': path does not exist.")
                continue
            print(f"    Processing split: {split}")

            try:
                class_names = sorted(os.listdir(split_path))
            except Exception as e:
                print(f"      ERROR: could not list {split_path}: {e}")
                continue

            for class_name in class_names:
                class_dir = os.path.join(split_path, class_name)
                if not os.path.isdir(class_dir):
                    continue

                dest_class_path = os.path.join(output_root, split, class_name)
                os.makedirs(dest_class_path, exist_ok=True)

                try:
                    files_in_class = sorted(
                        f for f in os.listdir(class_dir)
                        if os.path.isfile(os.path.join(class_dir, f))
                        and os.path.splitext(f)[1].lower() == '.wav')
                except Exception as e:
                    print(f"        ERROR: could not list {class_dir}: {e}")
                    continue

                print(f"      Processing class: {class_name} "
                      f"({len(files_in_class)} files)")

                for file_index, filename in enumerate(files_in_class):
                    src_file = os.path.join(class_dir, filename)

                    if file_index % 500 == 0 and file_index > 0:
                        print(f"        {file_index}/{len(files_in_class)}")

                    clip_index = 0
                    extension = os.path.splitext(filename)[1]
                    new_name = (f"{dataset_name}_{split}_{class_name}_"
                                f"{file_index}_{clip_index}{extension}")
                    dst_file = os.path.join(dest_class_path, new_name)

                    if os.path.exists(dst_file) and os.path.getsize(dst_file) > 0:
                        continue
                    shutil.copy2(src_file, dst_file)

                n = len(files_in_class)
                dataset_total += n
                total_copied += n
                per_class[(split, class_name)] = per_class.get(
                    (split, class_name), 0) + n

        per_dataset[dataset_name] = dataset_total

    print()
    print("Files per dataset:")
    for name in dataset_folders:
        print(f"  {name:<12} {per_dataset.get(name, 0):>7}")

    classes = sorted({c for _, c in per_class})
    print()
    print(f"{'split':<12}" + "".join(f"{c:>16}" for c in classes) + f"{'total':>9}")
    for split in splits:
        row = "".join(f"{per_class.get((split, c), 0):>16}" for c in classes)
        print(f"{split:<12}{row}"
              f"{sum(per_class.get((split, c), 0) for c in classes):>9}")
    print(f"{'TOTAL':<12}"
          + "".join(f"{sum(per_class.get((s, c), 0) for s in splits):>16}"
                    for c in classes)
          + f"{total_copied:>9}")

    print()
    print(f"Successfully created cumulative dataset at: "
          f"{os.path.abspath(output_root)}")


root_source = "...../Outputs" #CHANGE THIS TO OUTPUTS FOLDER
create_cumulative_dataset(root_source)


##CREATE MEL DATASET

In [ ]:
#This script converts the wav dataset into mel spectrogram .npy arrays, mirroring
#the <split>/<class> folder structure: INPUT_DIR/<split>/<class>/*.wav ->
#OUTPUT_DIR/<split>/<class>/*.npy

import sys
import time
from multiprocessing import Pool, cpu_count
from pathlib import Path

import numpy as np

# ======================= EDIT THESE =======================
INPUT_DIR = Path("Outputs/cumulative_dataset")   #CHANGE THIS: folder holding <split>/<class>/*.wav
OUTPUT_DIR = Path("Outputs/mel_dataset")         #CHANGE THIS: where the mirrored .npy folders will be written

SPLITS = ["train", "validation", "test"]

SR = 16000
N_FFT = 1024
WIN_LENGTH = 1024
HOP_LENGTH = 128
N_MELS = 128
FMIN = 0
FMAX = SR // 2
CLIP_SAMPLES = SR          # 1 second clips

LOG_SCALE = True
MINMAX_01 = True
DTYPE = np.float32

WORKERS = max(1, cpu_count() - 1)   # set to 1 if multiprocessing gives trouble in the notebook
# ==========================================================


def mel_from_wav(wav_path):
    import librosa

    y, _ = librosa.load(str(wav_path), sr=SR, mono=True)
    if len(y) < CLIP_SAMPLES:
        y = np.pad(y, (0, CLIP_SAMPLES - len(y)))
    else:
        y = y[:CLIP_SAMPLES]

    S = librosa.feature.melspectrogram(
        y=y, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH,
        window="hann", center=True, power=2.0,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX,
    )

    if LOG_SCALE:
        S = librosa.power_to_db(S, ref=np.max)

    if MINMAX_01:
        lo, hi = float(S.min()), float(S.max())
        S = (S - lo) / (hi - lo) if hi > lo else np.zeros_like(S)

    return S.astype(DTYPE)


def process_one(job):
    src, dst = job
    try:
        arr = mel_from_wav(src)
        np.save(dst, arr)
        return True, arr.shape
    except Exception as exc:
        return False, f"{src}: {exc}"


def collect_jobs():
    jobs = []
    per_split = {}
    for split in SPLITS:
        split_dir = INPUT_DIR / split
        if not split_dir.is_dir():
            print(f"  !! split folder missing: {split_dir}", flush=True)
            continue
        counts = {}
        for class_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
            out_dir = OUTPUT_DIR / split / class_dir.name
            out_dir.mkdir(parents=True, exist_ok=True)
            wavs = sorted(class_dir.glob("*.wav"))
            for w in wavs:
                jobs.append((w, out_dir / f"{w.stem}.npy"))
            counts[class_dir.name] = len(wavs)
        per_split[split] = counts
    return jobs, per_split


def main():
    try:
        import librosa
    except ImportError:
        sys.exit("this script needs librosa:  pip install librosa soundfile")

    if not INPUT_DIR.is_dir():
        raise SystemExit(f"input dir not found: {INPUT_DIR}")

    print(f"mel config: n_fft {N_FFT} | win {WIN_LENGTH} | hop {HOP_LENGTH} | "
          f"n_mels {N_MELS} | sr {SR}", flush=True)
    print(f"  -> {1 + CLIP_SAMPLES // HOP_LENGTH} frames per 1s clip "
          f"(arrays are {N_MELS} x {1 + CLIP_SAMPLES // HOP_LENGTH})", flush=True)
    print(f"  log_scale={LOG_SCALE}  minmax01={MINMAX_01}  dtype={np.dtype(DTYPE).name}\n",
          flush=True)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    jobs, per_split = collect_jobs()
    if not jobs:
        raise SystemExit(f"no .wav files found under {INPUT_DIR}")

    print("source clips:")
    for split, counts in per_split.items():
        total = sum(counts.values())
        detail = "  ".join(f"{c}={n}" for c, n in sorted(counts.items()))
        print(f"  {split:11} {total:>6}   {detail}", flush=True)
    print(f"  {'TOTAL':11} {len(jobs):>6}\n", flush=True)

    print(f"generating {len(jobs)} mel arrays on {WORKERS} workers...", flush=True)
    t0 = time.time()
    done, failed, shapes = 0, [], set()

    def handle(ok, info):
        if ok:
            shapes.add(info)
        else:
            failed.append(info)

    if WORKERS == 1:
        for job in jobs:
            ok, info = process_one(job)
            handle(ok, info)
            done += 1
            if done % 1000 == 0 or done == len(jobs):
                rate = done / max(time.time() - t0, 1e-9)
                print(f"  {done}/{len(jobs)}  ({rate:.0f} clips/s)", flush=True)
    else:
        with Pool(WORKERS) as pool:
            for ok, info in pool.imap_unordered(process_one, jobs, chunksize=32):
                handle(ok, info)
                done += 1
                if done % 1000 == 0 or done == len(jobs):
                    rate = done / max(time.time() - t0, 1e-9)
                    print(f"  {done}/{len(jobs)}  ({rate:.0f} clips/s)", flush=True)
    print(f"  finished in {time.time() - t0:.0f}s", flush=True)

    print(f"\narray shapes produced: {sorted(shapes)}", flush=True)
    if len(shapes) > 1:
        print("  !! more than one shape - check for clips that were not 1s", flush=True)
    if failed:
        print(f"  !! {len(failed)} clip(s) failed:", flush=True)
        for msg in failed[:10]:
            print(f"     {msg}", flush=True)

    n_npy = sum(1 for _ in OUTPUT_DIR.rglob("*.npy"))
    size_mb = sum(p.stat().st_size for p in OUTPUT_DIR.rglob("*.npy")) / 1e6
    print(f"\nwrote {n_npy} .npy files ({size_mb:.0f} MB) under {OUTPUT_DIR}", flush=True)
    print(f"  -> {OUTPUT_DIR}/<split>/<class>/*.npy")


if __name__ == "__main__":
    main()


##CREATE MCG DATASET

In [ ]:
#This script converts the wav dataset into 3-channel mel/CQT/gammatone .npy
#arrays (CHW), mirroring the <split>/<class> folder structure:
#INPUT_DIR/<split>/<class>/*.wav -> OUTPUT_DIR/<split>/<class>/*.npy

import sys
import time
from multiprocessing import Pool, cpu_count
from pathlib import Path

import numpy as np

# ======================= EDIT THESE =======================
INPUT_DIR = Path("..../Outputs/cumulative_dataset")   #CHANGE THIS: folder holding <split>/<class>/*.wav
OUTPUT_DIR = Path("..../Outputs/mcg_dataset")         #CHANGE THIS: where the mirrored .npy folders will be written

SPLITS = ["train", "validation", "test"]

SR = 16000
N_FFT = 1024
WIN_LENGTH = 1024
HOP_LENGTH = 128
CLIP_SAMPLES = SR                # 1 second clips
N_FRAMES = 1 + CLIP_SAMPLES // HOP_LENGTH

N_MELS = 128
MEL_FMIN, MEL_FMAX = 0, SR // 2

CQT_N_BINS = 128
CQT_BINS_PER_OCTAVE = 16
CQT_FMIN = 30.0

N_GAMMA = 128
GAMMA_FMIN, GAMMA_FMAX = 30.0, SR // 2
GAMMA_ORDER = 4

TOP_DB = 80.0
DTYPE = np.float32

WORKERS = max(1, cpu_count() - 1)   # set to 1 if multiprocessing gives trouble in the notebook
# ==========================================================


def _erb(f):
    return 24.7 * (4.37 * f / 1000.0 + 1.0)


def _erb_space(fmin, fmax, n):
    ear_q, min_bw = 9.26449, 24.7
    idx = np.arange(1, n + 1)
    cf = -(ear_q * min_bw) + np.exp(
        idx * (-np.log(fmax + ear_q * min_bw) + np.log(fmin + ear_q * min_bw)) / n
    ) * (fmax + ear_q * min_bw)
    return cf[::-1]


def gammatone_weights():
    freqs = np.fft.rfftfreq(N_FFT, 1.0 / SR)
    cf = _erb_space(GAMMA_FMIN, GAMMA_FMAX, N_GAMMA)
    bw = 1.019 * _erb(cf)
    w = (1.0 + ((freqs[None, :] - cf[:, None]) / bw[:, None]) ** 2) ** (-GAMMA_ORDER / 2.0)
    w /= np.maximum(w.max(axis=1, keepdims=True), 1e-10)
    return w


_GAMMA_W = None


def _gamma_w():
    global _GAMMA_W
    if _GAMMA_W is None:
        _GAMMA_W = gammatone_weights()
    return _GAMMA_W


def to_unit_db(power, ref_max=True):
    import librosa
    db = librosa.power_to_db(np.maximum(power, 1e-10),
                             ref=np.max if ref_max else 1.0, top_db=TOP_DB)
    lo, hi = float(db.min()), float(db.max())
    if hi <= lo:
        return np.zeros_like(db)
    return (db - lo) / (hi - lo)


def mcg_from_wav(wav_path):
    import librosa

    y, _ = librosa.load(str(wav_path), sr=SR, mono=True)
    if len(y) < CLIP_SAMPLES:
        y = np.pad(y, (0, CLIP_SAMPLES - len(y)))
    else:
        y = y[:CLIP_SAMPLES]

    stft = librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LENGTH,
                        win_length=WIN_LENGTH, window="hann", center=True)
    power = np.abs(stft) ** 2

    mel = librosa.feature.melspectrogram(S=power, sr=SR, n_mels=N_MELS,
                                         fmin=MEL_FMIN, fmax=MEL_FMAX)
    gam = _gamma_w() @ power

    cqt = np.abs(librosa.cqt(y, sr=SR, hop_length=HOP_LENGTH, fmin=CQT_FMIN,
                             n_bins=CQT_N_BINS,
                             bins_per_octave=CQT_BINS_PER_OCTAVE)) ** 2

    t = min(mel.shape[1], cqt.shape[1], gam.shape[1])
    mel, cqt, gam = mel[:, :t], cqt[:, :t], gam[:, :t]

    chans = [to_unit_db(mel), to_unit_db(cqt), to_unit_db(gam)]
    return np.stack(chans, axis=0).astype(DTYPE)


def process_one(job):
    src, dst = job
    try:
        arr = mcg_from_wav(src)
        np.save(dst, arr)
        return True, arr.shape
    except Exception as exc:
        return False, f"{src}: {exc}"


def collect_jobs():
    jobs, per_split = [], {}
    for split in SPLITS:
        split_dir = INPUT_DIR / split
        if not split_dir.is_dir():
            print(f"  !! split folder missing: {split_dir}", flush=True)
            continue
        counts = {}
        for class_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
            out_dir = OUTPUT_DIR / split / class_dir.name
            out_dir.mkdir(parents=True, exist_ok=True)
            wavs = sorted(class_dir.glob("*.wav"))
            for w in wavs:
                jobs.append((w, out_dir / f"{w.stem}.npy"))
            counts[class_dir.name] = len(wavs)
        per_split[split] = counts
    return jobs, per_split


def main():
    try:
        import librosa
    except ImportError:
        sys.exit("this script needs librosa:  pip install librosa soundfile")

    if not INPUT_DIR.is_dir():
        raise SystemExit(f"input dir not found: {INPUT_DIR}")

    print(f"STFT: n_fft {N_FFT} | win {WIN_LENGTH} | hop {HOP_LENGTH} | sr {SR}", flush=True)
    print(f"  ch0 mel   : {N_MELS} bands, {MEL_FMIN}-{MEL_FMAX} Hz", flush=True)
    print(f"  ch1 cqt   : {CQT_N_BINS} bins, {CQT_BINS_PER_OCTAVE}/octave from "
          f"{CQT_FMIN:.0f} Hz -> top {CQT_FMIN * 2 ** (CQT_N_BINS / CQT_BINS_PER_OCTAVE):.0f} Hz",
          flush=True)
    print(f"  ch2 gamma : {N_GAMMA} ERB filters, {GAMMA_FMIN:.0f}-{GAMMA_FMAX} Hz", flush=True)
    print(f"  -> arrays are (3, {N_MELS}, {N_FRAMES}) CHW, {np.dtype(DTYPE).name}, "
          f"each channel dB + min-max to [0,1]\n", flush=True)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    jobs, per_split = collect_jobs()
    if not jobs:
        raise SystemExit(f"no .wav files found under {INPUT_DIR}")

    print("source clips:")
    for split, counts in per_split.items():
        detail = "  ".join(f"{c}={n}" for c, n in sorted(counts.items()))
        print(f"  {split:11} {sum(counts.values()):>6}   {detail}", flush=True)
    print(f"  {'TOTAL':11} {len(jobs):>6}\n", flush=True)

    est_gb = len(jobs) * 3 * N_MELS * N_FRAMES * np.dtype(DTYPE).itemsize / 1e9
    print(f"generating {len(jobs)} arrays on {WORKERS} workers "
          f"(~{est_gb:.2f} GB expected; CQT is the slow channel)...", flush=True)
    t0 = time.time()
    done, failed, shapes = 0, [], set()

    def handle(ok, info):
        if ok:
            shapes.add(info)
        else:
            failed.append(info)

    def progress():
        el = time.time() - t0
        rate = done / max(el, 1e-9)
        eta = (len(jobs) - done) / max(rate, 1e-9)
        print(f"  {done}/{len(jobs)}  ({rate:.1f} clips/s, eta {eta/60:.1f} min)",
              flush=True)

    if WORKERS == 1:
        for job in jobs:
            ok, info = process_one(job)
            handle(ok, info)
            done += 1
            if done % 500 == 0 or done == len(jobs):
                progress()
    else:
        with Pool(WORKERS) as pool:
            for ok, info in pool.imap_unordered(process_one, jobs, chunksize=16):
                handle(ok, info)
                done += 1
                if done % 500 == 0 or done == len(jobs):
                    progress()
    print(f"  finished in {(time.time() - t0)/60:.1f} min", flush=True)

    print(f"\narray shapes produced: {sorted(shapes)}", flush=True)
    if len(shapes) > 1:
        print("  !! more than one shape - inspect before training", flush=True)
    if failed:
        print(f"  !! {len(failed)} clip(s) failed:", flush=True)
        for msg in failed[:10]:
            print(f"     {msg}", flush=True)

    n_npy = sum(1 for _ in OUTPUT_DIR.rglob("*.npy"))
    size_gb = sum(p.stat().st_size for p in OUTPUT_DIR.rglob("*.npy")) / 1e9
    print(f"\nwrote {n_npy} .npy files ({size_gb:.2f} GB) under {OUTPUT_DIR}", flush=True)
    print(f"  -> {OUTPUT_DIR}/<split>/<class>/*.npy   (3, {N_MELS}, {N_FRAMES}) CHW")


if __name__ == "__main__":
    main()


##CREATE MFCC DATASET

In [ ]:
#This script converts the wav dataset into standardized MFCC feature vectors
#(mean + std over time per clip). Reads INPUT_DIR/<split>/<class>/*.wav and
#writes X/y .npy arrays plus the scaler and metadata into OUTPUT_DIR.

import json
import sys
import time
from multiprocessing import Pool, cpu_count
from pathlib import Path

import numpy as np

# ======================= EDIT THESE =======================
INPUT_DIR = Path("..../Outputs/cumulative_dataset")   #CHANGE THIS: folder holding <split>/<class>/*.wav
OUTPUT_DIR = Path("..../Outputs/mfcc_dataset")        #CHANGE THIS: where the X/y arrays and metadata will be written

SPLITS = ["train", "validation", "test"]
FIT_SPLIT = "train"      # the split the scaler is fit on

SR = 16000
N_FFT = 1024
WIN_LENGTH = 1024
HOP_LENGTH = 128
CLIP_SAMPLES = SR        # 1 second clips

N_MELS = 128
FMIN, FMAX = 0, 8000

N_MFCC = 20
DCT_TYPE = 2
DCT_NORM = "ortho"
LIFTER = 0
DROP_C0 = False

WORKERS = max(1, cpu_count() - 1)   # set to 1 if multiprocessing gives trouble in the notebook
# ==========================================================


def find_split_root(base):
    base = Path(base)
    if not base.exists():
        return None
    for d in [base] + sorted(p for p in base.iterdir() if p.is_dir()):
        if (d / "train").is_dir() and (d / "validation").is_dir() \
                and next((d / "validation").rglob("*.wav"), None) is not None:
            return d
    return None


def mfcc_features(wav_path):
    import librosa

    y, _ = librosa.load(str(wav_path), sr=SR, mono=True)
    if len(y) < CLIP_SAMPLES:
        y = np.pad(y, (0, CLIP_SAMPLES - len(y)))
    else:
        y = y[:CLIP_SAMPLES]

    m = librosa.feature.mfcc(
        y=y, sr=SR, n_mfcc=N_MFCC, dct_type=DCT_TYPE, norm=DCT_NORM, lifter=LIFTER,
        n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH,
        window="hann", center=True, n_mels=N_MELS, fmin=FMIN, fmax=FMAX,
    )

    if DROP_C0:
        m = m[1:]
    return np.concatenate([m.mean(axis=1), m.std(axis=1)]).astype(np.float32)


def process_one(job):
    path, label = job
    try:
        return True, (mfcc_features(path), label, path.name)
    except Exception as exc:
        return False, f"{path}: {exc}"


def collect_jobs(split_dir, class_to_idx):
    jobs, counts = [], {}
    for cls, idx in class_to_idx.items():
        cdir = split_dir / cls
        wavs = sorted(cdir.glob("*.wav")) if cdir.is_dir() else []
        if not cdir.is_dir():
            print(f"  !! {split_dir.name}: no folder for class '{cls}'", flush=True)
        for w in wavs:
            jobs.append((w, idx))
        counts[cls] = len(wavs)
    return jobs, counts


def feature_names():
    idx = range(1, N_MFCC) if DROP_C0 else range(N_MFCC)
    return [f"mfcc{i}_mean" for i in idx] + [f"mfcc{i}_std" for i in idx]


def main():
    try:
        import librosa
    except ImportError:
        sys.exit("this script needs librosa:  pip install librosa soundfile")

    n_feat = 2 * (N_MFCC - 1 if DROP_C0 else N_MFCC)
    print(f"MFCC: n_mfcc {N_MFCC} (C0 {'dropped' if DROP_C0 else 'kept'}) | "
          f"dct {DCT_TYPE}/{DCT_NORM} | lifter {LIFTER}", flush=True)
    print(f"STFT: n_fft {N_FFT} | win {WIN_LENGTH} | hop {HOP_LENGTH} | sr {SR}", flush=True)
    print(f"filterbank: {N_MELS} mels, {FMIN}-{FMAX} Hz", flush=True)
    print(f"aggregation: mean + std over time -> {n_feat} features per clip\n", flush=True)

    root = find_split_root(INPUT_DIR)
    if root is None:
        raise SystemExit(f"could not find train/validation wav splits under {INPUT_DIR}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    classes = sorted(d.name for d in (root / "train").iterdir() if d.is_dir())
    class_to_idx = {c: i for i, c in enumerate(classes)}
    print(f"classes: {classes}\n", flush=True)

    raw, labels, names = {}, {}, {}
    for split in SPLITS:
        sdir = root / split
        if not sdir.is_dir():
            print(f"  !! split folder missing: {sdir}", flush=True)
            continue
        jobs, counts = collect_jobs(sdir, class_to_idx)
        if not jobs:
            print(f"  !! no wavs in {sdir}", flush=True)
            continue

        detail = "  ".join(f"{c}={n}" for c, n in sorted(counts.items()))
        print(f"[{split}] {len(jobs)} clips   {detail}", flush=True)

        t0, feats, labs, fnames, failed = time.time(), [], [], [], []

        def handle(ok, info):
            if ok:
                v, lab, nm = info
                feats.append(v); labs.append(lab); fnames.append(nm)
            else:
                failed.append(info)

        if WORKERS == 1:
            for job in jobs:
                ok, info = process_one(job)
                handle(ok, info)
        else:
            with Pool(WORKERS) as pool:
                for ok, info in pool.imap(process_one, jobs, chunksize=64):
                    handle(ok, info)

        raw[split] = np.stack(feats).astype(np.float32)
        labels[split] = np.asarray(labs, dtype=np.int64)
        names[split] = fnames
        print(f"    -> {raw[split].shape} in {time.time() - t0:.0f}s", flush=True)
        if failed:
            print(f"    !! {len(failed)} clip(s) failed:", flush=True)
            for msg in failed[:5]:
                print(f"       {msg}", flush=True)

    if FIT_SPLIT not in raw:
        raise SystemExit(f"'{FIT_SPLIT}' split produced no features; cannot fit scaler")

    mean = raw[FIT_SPLIT].mean(axis=0)
    scale = raw[FIT_SPLIT].std(axis=0)
    scale[scale < 1e-8] = 1.0                            # guard constant features
    print(f"\nscaler fit on '{FIT_SPLIT}' ({raw[FIT_SPLIT].shape[0]} clips) "
          f"and applied to {', '.join(raw)}", flush=True)

    for split in raw:
        X = ((raw[split] - mean) / scale).astype(np.float32)
        np.save(OUTPUT_DIR / f"X_{split}.npy", X)
        np.save(OUTPUT_DIR / f"X_{split}_raw.npy", raw[split])
        np.save(OUTPUT_DIR / f"y_{split}.npy", labels[split])
        (OUTPUT_DIR / f"files_{split}.json").write_text(json.dumps(names[split]))
        print(f"  {split:11} X {X.shape}  mean {X.mean():+.3f}  std {X.std():.3f}",
              flush=True)

    np.save(OUTPUT_DIR / "scaler_mean.npy", mean.astype(np.float32))
    np.save(OUTPUT_DIR / "scaler_scale.npy", scale.astype(np.float32))
    (OUTPUT_DIR / "classes.json").write_text(json.dumps(classes))
    (OUTPUT_DIR / "metadata.json").write_text(json.dumps({
        "sr": SR, "n_fft": N_FFT, "win_length": WIN_LENGTH, "hop_length": HOP_LENGTH,
        "n_mels": N_MELS, "fmin": FMIN, "fmax": FMAX,
        "n_mfcc": N_MFCC, "dct_type": DCT_TYPE, "norm": DCT_NORM, "lifter": LIFTER,
        "drop_c0": DROP_C0, "aggregation": "mean+std over time",
        "n_features": n_feat, "feature_names": feature_names(),
        "classes": classes, "scaler_fit_split": FIT_SPLIT,
        "counts": {s: int(raw[s].shape[0]) for s in raw},
    }, indent=2))

    print(f"\ndone. features written to {OUTPUT_DIR}")
    print(f"  X = np.load('{OUTPUT_DIR}/X_train.npy')   # already standardized")
    print(f"  y = np.load('{OUTPUT_DIR}/y_train.npy')")


if __name__ == "__main__":
    main()
